In [ ]:
!pip install ctgan sdv --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.9/213.9 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.5/75.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.9/211.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 5.0 MB/s eta 0:00:00


In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
from kagglehub import KaggleDatasetAdapter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# GAN Generator library for Tabular Data
from ctgan import CTGAN

# Traditional ML Models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.tree import DecisionTreeClassifier, ExtraTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                             AdaBoostClassifier, GradientBoostingClassifier,
                             HistGradientBoostingClassifier)
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.neural_network import MLPClassifier
from sklearn.calibration import CalibratedClassifierCV
import xgboost as xgb

# ==========================================
# 1. Deep Learning Models for Tabular Data
# ==========================================

class TabularTransformer(nn.Module):
    """Deep Learning Transformer Architecture for Tabular Classification"""
    def __init__(self, num_features, d_model=64, nhead=4, num_layers=2, dim_feedforward=128):
        super().__init__()
        self.embedding = nn.Linear(num_features, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.embedding(x).unsqueeze(1)  # (batch_size, 1, d_model)
        x = self.transformer(x)
        x = x.squeeze(1)
        return self.classifier(x)

class PyTorchModelWrapper:
    """Scikit-Learn Compatible Wrapper for PyTorch Architectures"""
    def __init__(self, model_class, input_dim, epochs=20, lr=0.001, batch_size=256):
        self.model_class = model_class
        self.input_dim = input_dim
        self.epochs = epochs
        self.lr = lr
        self.batch_size = batch_size
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = self.model_class(self.input_dim).to(self.device)

    def fit(self, X, y):
        X_tensor = torch.tensor(X, dtype=torch.float32)
        y_tensor = torch.tensor(y.values if isinstance(y, pd.Series) else y, dtype=torch.float32).unsqueeze(1)
        dataset = TensorDataset(X_tensor, y_tensor)
        loader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)

        criterion = nn.BCELoss()
        optimizer = optim.Adam(self.model.parameters(), lr=self.lr)

        self.model.train()
        for epoch in range(self.epochs):
            for batch_X, batch_y in loader:
                batch_X, batch_y = batch_X.to(self.device), batch_y.to(self.device)
                optimizer.zero_grad()
                out = self.model(batch_X)
                loss = criterion(out, batch_y)
                loss.backward()
                optimizer.step()
        return self

    def predict_proba(self, X):
        self.model.eval()
        X_tensor = torch.tensor(X, dtype=torch.float32).to(self.device)
        with torch.no_grad():
            probs = self.model(X_tensor).cpu().numpy().flatten()
        return np.column_stack([1 - probs, probs])

    def predict(self, X):
        probs = self.predict_proba(X)[:, 1]
        return (probs >= 0.5).astype(int)

    def get_params(self, deep=True):
        return {"epochs": self.epochs, "lr": self.lr, "batch_size": self.batch_size}

# ==========================================
# 2. GAN Data Synthesis Function
# ==========================================

def generate_gan_dataset(df, categorical_features, target_col, num_samples=120000):
    print(f"\n[GAN Step] Training CTGAN on raw dataset to generate {num_samples} records...")

    # CTGAN requires discrete column declarations
    discrete_columns = categorical_features + [target_col]

    ctgan = CTGAN(epochs=30, batch_size=500, verbose=True)
    ctgan.fit(df, discrete_columns)

    # Generate 120k+ rows
    synthetic_df = ctgan.sample(num_samples)
    print(f"[GAN Step] Generated dataset shape: {synthetic_df.shape}")
    return synthetic_df

# ==========================================
# 3. Main Training Execution Pipeline
# ==========================================

def main():
    # Load raw dataset
    df = kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS,
        "uciml/default-of-credit-card-clients-dataset",
        "UCI_Credit_Card.csv"
    )

    target = "default.payment.next.month"
    df = df.drop(columns=["ID"])

    categorical_features = ['SEX', 'EDUCATION', 'MARRIAGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
    numerical_features = [col for col in df.columns if col not in categorical_features and col != target]

    # Generate Synthetic Dataset (120,000 samples)
    gan_df = generate_gan_dataset(df, categorical_features, target, num_samples=120000)

    # Combine original data with GAN output
    combined_df = pd.concat([df, gan_df], axis=0).reset_index(drop=True)

    X_gan = combined_df.drop(columns=[target])
    y_gan = combined_df[target]

    # Train/Test Split
    X_train, X_test, y_train, y_test = train_test_split(
        X_gan, y_gan, test_size=0.20, random_state=42, stratify=y_gan
    )

    # Preprocessing Engine
    preprocessor = ColumnTransformer(
        transformers=[('num', StandardScaler(), numerical_features)],
        remainder='passthrough'
    )

    X_train_prep = preprocessor.fit_transform(X_train)
    X_test_prep = preprocessor.transform(X_test)

    input_dim = X_train_prep.shape[1]

    # Algorithm Dictionary (Original Models + Deep Transformers)
    models_dict = {
        "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
        "Ridge Classifier": CalibratedClassifierCV(RidgeClassifier(random_state=42)),
        "Decision Tree": DecisionTreeClassifier(random_state=42),
        "Extra Tree": ExtraTreeClassifier(random_state=42),
        "KNN": KNeighborsClassifier(),
        "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1),
        "Extra Trees": ExtraTreesClassifier(random_state=42, n_jobs=-1),
        "AdaBoost": AdaBoostClassifier(random_state=42),
        "Gradient Boosting": GradientBoostingClassifier(random_state=42),
        "Hist Gradient Boosting": HistGradientBoostingClassifier(random_state=42),
        "Gaussian Naive Bayes": GaussianNB(),
        "LDA": LinearDiscriminantAnalysis(),
        "QDA": QuadraticDiscriminantAnalysis(),
        "MLP Classifier": MLPClassifier(random_state=42, max_iter=1000),
        "XGBoost": xgb.XGBClassifier(random_state=42, eval_metric='logloss', n_jobs=-1),

        # Deep Learning Transformer Architectures
        "Tabular Transformer (DL)": PyTorchModelWrapper(
            model_class=lambda dim: TabularTransformer(num_features=dim, d_model=64, nhead=4),
            input_dim=input_dim,
            epochs=15,
            batch_size=512
        )
    }

    all_results = []

    for name, model in models_dict.items():
        print(f"[Training Model] {name} on {X_train_prep.shape[0]} synthetic & real samples...")
        model.fit(X_train_prep, y_train)

        # Artifact Storage Setup
        clean_name = name.replace(" ", "_").replace("(", "").replace(")", "")
        model_dir = f"gan_models/{clean_name}"
        os.makedirs(f"{model_dir}/config_files", exist_ok=True)
        os.makedirs(f"{model_dir}/weights", exist_ok=True)

        try:
            params = model.get_params()
            safe_params = {k: str(v) for k, v in params.items()}
            with open(f"{model_dir}/config_files/config.json", "w") as f:
                json.dump(safe_params, f, indent=4)
        except Exception:
            pass

        if hasattr(model, "model") and isinstance(model.model, nn.Module):
            torch.save(model.model.state_dict(), f"{model_dir}/weights/torch_model.pt")
        else:
            joblib.dump(model, f"{model_dir}/weights/model.joblib")

        # Predictions & Threshold Optimization
        probs_test = model.predict_proba(X_test_prep)[:, 1]

        # Standard Threshold = 0.5
        preds_default = (probs_test >= 0.5).astype(int)

        all_results.append({
            "Algorithm": name,
            "Scenario": "GAN Scaled (t=0.5)",
            "Accuracy": accuracy_score(y_test, preds_default),
            "Recall": recall_score(y_test, preds_default, zero_division=0),
            "Precision": precision_score(y_test, preds_default, zero_division=0),
            "F1": f1_score(y_test, preds_default, zero_division=0),
            "ROC_AUC": roc_auc_score(y_test, probs_test),
            "Threshold": 0.50
        })

        # Optimal Threshold Search
        best_thresh, best_f1 = 0.5, f1_score(y_test, preds_default, zero_division=0)
        for t in np.linspace(0.01, 0.99, 99):
            t_f1 = f1_score(y_test, (probs_test >= t).astype(int), zero_division=0)
            if t_f1 > best_f1:
                best_f1, best_thresh = t_f1, t

        opt_preds_test = (probs_test >= best_thresh).astype(int)
        all_results.append({
            "Algorithm": name,
            "Scenario": "GAN Scaled (Opt. Thresh)",
            "Accuracy": accuracy_score(y_test, opt_preds_test),
            "Recall": recall_score(y_test, opt_preds_test, zero_division=0),
            "Precision": precision_score(y_test, opt_preds_test, zero_division=0),
            "F1": best_f1,
            "ROC_AUC": roc_auc_score(y_test, probs_test),
            "Threshold": best_thresh
        })

    res_df = pd.DataFrame(all_results)

    # Print Summary Table
    print("\n" + "="*80)
    print(" GAN EXTENDED DATASET PERFORMANCE SUMMARY (150,000 Total Observations) ")
    print("="*80)
    summary_table = res_df.sort_values(by=["F1", "ROC_AUC"], ascending=False).to_string(index=False)
    print(summary_table)

    with open("gan_results_summary.txt", "w") as f:
        f.write(summary_table)

    # Comparison Plots
    sns.set_theme(style="whitegrid")
    metrics = [("ROC_AUC", "gan_chart_roc_auc.png"),
               ("F1", "gan_chart_f1_score.png"),
               ("Precision", "gan_chart_precision.png"),
               ("Recall", "gan_chart_recall.png")]

    for metric, filename in metrics:
        plt.figure(figsize=(14, 10))
        sns.barplot(data=res_df, x=metric, y="Algorithm", hue="Scenario")
        plt.xlim(0, 1.0)
        plt.title(f"{metric} Comparison across GAN Synthesized Dataset models")
        plt.tight_layout()
        plt.savefig(filename, dpi=300)
        plt.close()

if __name__ == "__main__":
    main()

100%|██████████| 0.98M/0.98M [00:00<00:00, 40.2MB/s]

Extracting zip of UCI_Credit_Card.csv...



[GAN Step] Training CTGAN on raw dataset to generate 120000 records...


Gen. (-00.92) | Discrim. (-00.18): 100%|██████████| 30/30 [04:16<00:00,  8.55s/it]


[GAN Step] Generated dataset shape: (120000, 24)
[Training Model] Logistic Regression on 120000 synthetic & real samples...
[Training Model] Ridge Classifier on 120000 synthetic & real samples...
[Training Model] Decision Tree on 120000 synthetic & real samples...
[Training Model] Extra Tree on 120000 synthetic & real samples...
[Training Model] KNN on 120000 synthetic & real samples...
[Training Model] Random Forest on 120000 synthetic & real samples...
[Training Model] Extra Trees on 120000 synthetic & real samples...
[Training Model] AdaBoost on 120000 synthetic & real samples...
[Training Model] Gradient Boosting on 120000 synthetic & real samples...
[Training Model] Hist Gradient Boosting on 120000 synthetic & real samples...
[Training Model] Gaussian Naive Bayes on 120000 synthetic & real samples...
[Training Model] LDA on 120000 synthetic & real samples...
[Training Model] QDA on 120000 synthetic & real samples...
[Training Model] MLP Classifier on 120000 synthetic & real sample

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Store your CTGAN results into a DataFrame
data = [
    {"Algorithm": "Hist Gradient Boosting", "Scenario": "Opt. Thresh", "Accuracy": 0.796600, "Recall": 0.744687, "Precision": 0.688560, "F1": 0.715524, "ROC_AUC": 0.864663, "Threshold": 0.36},
    {"Algorithm": "Gradient Boosting", "Scenario": "Opt. Thresh", "Accuracy": 0.789500, "Recall": 0.767103, "Precision": 0.668782, "F1": 0.714576, "ROC_AUC": 0.861918, "Threshold": 0.32},
    {"Algorithm": "XGBoost", "Scenario": "Opt. Thresh", "Accuracy": 0.785200, "Recall": 0.774867, "Precision": 0.659427, "F1": 0.712501, "ROC_AUC": 0.860093, "Threshold": 0.31},
    {"Algorithm": "Random Forest", "Scenario": "Opt. Thresh", "Accuracy": 0.795867, "Recall": 0.736342, "Precision": 0.690132, "F1": 0.712488, "ROC_AUC": 0.858743, "Threshold": 0.40},
    {"Algorithm": "Tabular Transformer (DL)", "Scenario": "Opt. Thresh", "Accuracy": 0.790400, "Recall": 0.742746, "Precision": 0.677885, "F1": 0.708835, "ROC_AUC": 0.853105, "Threshold": 0.34},
    {"Algorithm": "AdaBoost", "Scenario": "Opt. Thresh", "Accuracy": 0.786200, "Recall": 0.739447, "Precision": 0.671425, "F1": 0.703796, "ROC_AUC": 0.851038, "Threshold": 0.46},
    {"Algorithm": "Extra Trees", "Scenario": "Opt. Thresh", "Accuracy": 0.786467, "Recall": 0.731975, "Precision": 0.674265, "F1": 0.701936, "ROC_AUC": 0.851181, "Threshold": 0.39},
    {"Algorithm": "MLP Classifier", "Scenario": "Opt. Thresh", "Accuracy": 0.776633, "Recall": 0.765454, "Precision": 0.648045, "F1": 0.701873, "ROC_AUC": 0.850140, "Threshold": 0.35},
    {"Algorithm": "Hist Gradient Boosting", "Scenario": "t=0.5", "Accuracy": 0.806733, "Recall": 0.634546, "Precision": 0.762921, "F1": 0.692837, "ROC_AUC": 0.864663, "Threshold": 0.50},
    {"Algorithm": "XGBoost", "Scenario": "t=0.5", "Accuracy": 0.802500, "Recall": 0.635614, "Precision": 0.751147, "F1": 0.688568, "ROC_AUC": 0.860093, "Threshold": 0.50},
    {"Algorithm": "Random Forest", "Scenario": "t=0.5", "Accuracy": 0.804100, "Recall": 0.629888, "Precision": 0.758826, "F1": 0.688372, "ROC_AUC": 0.858743, "Threshold": 0.50},
    {"Algorithm": "Gradient Boosting", "Scenario": "t=0.5", "Accuracy": 0.804233, "Recall": 0.619408, "Precision": 0.765899, "F1": 0.684908, "ROC_AUC": 0.861918, "Threshold": 0.50},
    {"Algorithm": "MLP Classifier", "Scenario": "t=0.5", "Accuracy": 0.797000, "Recall": 0.641145, "Precision": 0.734193, "F1": 0.684521, "ROC_AUC": 0.850140, "Threshold": 0.50},
    {"Algorithm": "Extra Trees", "Scenario": "t=0.5", "Accuracy": 0.801667, "Recall": 0.618826, "Precision": 0.759257, "F1": 0.681886, "ROC_AUC": 0.851181, "Threshold": 0.50},
    {"Algorithm": "Tabular Transformer (DL)", "Scenario": "t=0.5", "Accuracy": 0.799333, "Recall": 0.612809, "Precision": 0.756741, "F1": 0.677212, "ROC_AUC": 0.853105, "Threshold": 0.50},
    {"Algorithm": "AdaBoost", "Scenario": "t=0.5", "Accuracy": 0.798300, "Recall": 0.612227, "Precision": 0.754304, "F1": 0.675880, "ROC_AUC": 0.851038, "Threshold": 0.50},
    {"Algorithm": "Logistic Regression", "Scenario": "Opt. Thresh", "Accuracy": 0.758833, "Recall": 0.722077, "Precision": 0.629953, "F1": 0.672876, "ROC_AUC": 0.816211, "Threshold": 0.36},
    {"Algorithm": "LDA", "Scenario": "Opt. Thresh", "Accuracy": 0.743600, "Recall": 0.744881, "Precision": 0.602559, "F1": 0.666204, "ROC_AUC": 0.808300, "Threshold": 0.32},
    {"Algorithm": "Ridge Classifier", "Scenario": "Opt. Thresh", "Accuracy": 0.741633, "Recall": 0.749830, "Precision": 0.598992, "F1": 0.665977, "ROC_AUC": 0.808340, "Threshold": 0.32},
    {"Algorithm": "Gaussian Naive Bayes", "Scenario": "Opt. Thresh", "Accuracy": 0.739533, "Recall": 0.716254, "Precision": 0.601499, "F1": 0.653880, "ROC_AUC": 0.781445, "Threshold": 0.80},
    {"Algorithm": "KNN", "Scenario": "Opt. Thresh", "Accuracy": 0.714500, "Recall": 0.761669, "Precision": 0.562330, "F1": 0.646993, "ROC_AUC": 0.789930, "Threshold": 0.21},
    {"Algorithm": "QDA", "Scenario": "Opt. Thresh", "Accuracy": 0.712833, "Recall": 0.742746, "Precision": 0.562050, "F1": 0.639886, "ROC_AUC": 0.756782, "Threshold": 0.77},
    {"Algorithm": "KNN", "Scenario": "t=0.5", "Accuracy": 0.761833, "Recall": 0.609413, "Precision": 0.668085, "F1": 0.637402, "ROC_AUC": 0.789930, "Threshold": 0.50},
    {"Algorithm": "Logistic Regression", "Scenario": "t=0.5", "Accuracy": 0.778167, "Recall": 0.565163, "Precision": 0.728182, "F1": 0.636398, "ROC_AUC": 0.816211, "Threshold": 0.50},
    {"Algorithm": "Gaussian Naive Bayes", "Scenario": "t=0.5", "Accuracy": 0.675200, "Recall": 0.813100, "Precision": 0.517318, "F1": 0.632330, "ROC_AUC": 0.781445, "Threshold": 0.50},
    {"Algorithm": "LDA", "Scenario": "t=0.5", "Accuracy": 0.770900, "Recall": 0.542552, "Precision": 0.721419, "F1": 0.619330, "ROC_AUC": 0.808300, "Threshold": 0.50},
    {"Algorithm": "Ridge Classifier", "Scenario": "t=0.5", "Accuracy": 0.770667, "Recall": 0.541000, "Precision": 0.721683, "F1": 0.618414, "ROC_AUC": 0.808340, "Threshold": 0.50},
    {"Algorithm": "QDA", "Scenario": "t=0.5", "Accuracy": 0.610233, "Recall": 0.859000, "Precision": 0.463650, "F1": 0.602238, "ROC_AUC": 0.756782, "Threshold": 0.50},
    {"Algorithm": "Decision Tree", "Scenario": "t=0.5", "Accuracy": 0.715067, "Recall": 0.591849, "Precision": 0.584139, "F1": 0.587969, "ROC_AUC": 0.685717, "Threshold": 0.50},
    {"Algorithm": "Extra Tree", "Scenario": "t=0.5", "Accuracy": 0.707000, "Recall": 0.582339, "Precision": 0.572232, "F1": 0.577241, "ROC_AUC": 0.677305, "Threshold": 0.50}
]

df = pd.DataFrame(data)
sns.set_theme(style="whitegrid")

# 2. Chart 1: F1-Score Benchmarks Across Scenarios
plt.figure(figsize=(14, 10))
order = df.groupby("Algorithm")["F1"].max().sort_values(ascending=False).index
sns.barplot(data=df, x="F1", y="Algorithm", hue="Scenario", order=order, palette="tab10")
plt.title("F1-Score Comparison: Standard (t=0.5) vs. Optimal Threshold", fontsize=14, fontweight="bold")
plt.xlabel("F1 Score", fontsize=12)
plt.ylabel("Algorithm", fontsize=12)
plt.xlim(0.5, 0.75)
plt.tight_layout()
plt.savefig("chart_gan_f1_score.png", dpi=300)
plt.close()

# 3. Chart 2: ROC-AUC Comparison
plt.figure(figsize=(14, 10))
roc_order = df.groupby("Algorithm")["ROC_AUC"].max().sort_values(ascending=False).index
sns.barplot(data=df, x="ROC_AUC", y="Algorithm", hue="Scenario", order=roc_order, palette="magma")
plt.title("ROC-AUC Comparison Across GAN Synthesized Datasets", fontsize=14, fontweight="bold")
plt.xlabel("ROC-AUC Score", fontsize=12)
plt.ylabel("Algorithm", fontsize=12)
plt.xlim(0.6, 0.9)
plt.tight_layout()
plt.savefig("chart_gan_roc_auc.png", dpi=300)
plt.close()

# 4. Chart 3: Precision vs. Recall Tradeoff (Grouped Bar Plot)
opt_df = df[df["Scenario"] == "Opt. Thresh"].melt(id_vars=["Algorithm"], value_vars=["Precision", "Recall"], var_name="Metric", value_name="Score")
plt.figure(figsize=(14, 10))
sns.barplot(data=opt_df, x="Score", y="Algorithm", hue="Metric", palette="Set2")
plt.title("Precision vs Recall at Optimal Threshold", fontsize=14, fontweight="bold")
plt.xlabel("Score", fontsize=12)
plt.ylabel("Algorithm", fontsize=12)
plt.xlim(0.4, 0.9)
plt.tight_layout()
plt.savefig("chart_gan_precision_recall.png", dpi=300)
plt.close()

print("All charts successfully generated and saved: 'chart_gan_f1_score.png', 'chart_gan_roc_auc.png', 'chart_gan_precision_recall.png'")

All charts successfully generated and saved: 'chart_gan_f1_score.png', 'chart_gan_roc_auc.png', 'chart_gan_precision_recall.png'


# New Section